In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

sys.path.insert(0, str(Path("..").resolve()))

In [ ]:
DATA_DIR = Path("../data")

IDEAS_PATH    = DATA_DIR / "ideas_validation_haiku_prompt_v1.json"
EMBEDDINGS_OUT = DATA_DIR / "idea_embeddings.npy"
METADATA_OUT   = DATA_DIR / "idea_metadata.csv"

ideas_data = json.loads(IDEAS_PATH.read_text(encoding="utf-8"))

rows = []
for book_id, book in ideas_data.items():
    for idea in book.get("ideas", []):
        rows.append({
            "book_id":  int(book_id),
            "title":    book["title"],
            "author":   book.get("author", ""),
            "idea":     idea["idea"],
            "stance":   idea["stance"],
            "strength": idea["strength"],
        })

meta_df = pd.DataFrame(rows)
print(f"{len(meta_df)} ideas from {meta_df['book_id'].nunique()} books")
meta_df.head()

In [3]:
model = SentenceTransformer("ai-forever/FRIDA")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/509 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/823 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [4]:
PREFIX = "paraphrase: "

In [5]:
texts = [PREFIX + idea for idea in meta_df["idea"].tolist()]

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(f"Embeddings shape: {embeddings.shape}")

Batches:   0%|          | 0/219 [00:00<?, ?it/s]

Embeddings shape: (13996, 1536)


In [ ]:
np.save(EMBEDDINGS_OUT, embeddings)
meta_df.to_csv(METADATA_OUT, index=False)

In [9]:
from sklearn.cluster import AgglomerativeClustering

N_CLUSTERS = 300  #потом определить 

clusterer = AgglomerativeClustering(
    n_clusters=N_CLUSTERS,
    metric="cosine",
    linkage="average",
)
labels = clusterer.fit_predict(embeddings)

meta_df["cluster"] = labels
meta_df["cluster"].value_counts().head(20)


cluster
45     1332
120     864
31      632
15      562
56      486
106     452
17      395
81      369
52      367
11      306
91      289
174     274
34      267
58      260
66      199
57      198
13      191
4       190
14      183
37      178
Name: count, dtype: int64

In [10]:
N_SAMPLES = 5

for cluster_id in sorted(meta_df["cluster"].unique()):
    sample = meta_df[meta_df["cluster"] == cluster_id]["idea"].sample(min(N_SAMPLES, len(meta_df[meta_df["cluster"] == cluster_id])), random_state=42).tolist()
    print(f"\n--- Cluster {cluster_id} ---")
    for idea in sample:
        print(f"  • {idea}")



--- Cluster 0 ---
  • Теодицея: невозможность оправдания Бога перед лицом страданий невинных детей
  • Если Бога нет, то всё позволено
  • Идея «всё дозволено» как логическое следствие атеизма
  • теодицея: оправдание путей Бога перед людьми
  • аргумент от сродства души с вечными идеями

--- Cluster 1 ---
  • Сила слов: как неосторожно сказанное может разрушить близкие отношения
  • непредвиденные последствия случайного вовлечения в чужие интриги
  • Долгосрочные последствия первого контакта для обеих сторон
  • Недосказанность и секреты внутри команды как источник конфликта
  • Парадокс первого контакта: опасность или возможность

--- Cluster 2 ---
  • Технологические волны как движущая сила смены общественных формаций
  • научно-технический прогресс на службе у деспотизма
  • Кибербезопасность как стратегический приоритет государств и корпораций
  • Детерминизм технологического развития: неизбежен ли прогресс техники
  • Технологический прогресс как фактор социальной атомизации

--

Single linkeage, complete

In [11]:
from sklearn.cluster import AgglomerativeClustering

SIM_THRESHOLD = 0.9  

clusterer = AgglomerativeClustering(
    n_clusters=None,
    metric="cosine",
    linkage="single",
    distance_threshold=1 - SIM_THRESHOLD,
)
labels = clusterer.fit_predict(embeddings)

meta_df["cluster"] = labels
print(f"{clusterer.n_clusters_} clusters found")
meta_df["cluster"].value_counts().head(20)


11136 clusters found


cluster
38      197
72      172
23       85
35       82
257      79
83       68
262      38
10       38
230      36
14       33
1345     27
61       26
56       23
51       22
318      19
676      18
167      18
338      17
19       17
665      17
Name: count, dtype: int64

In [12]:
N_SAMPLES = 5

for cluster_id in sorted(meta_df["cluster"].unique()):
    sample = meta_df[meta_df["cluster"] == cluster_id]["idea"].sample(min(N_SAMPLES, len(meta_df[meta_df["cluster"] == cluster_id])), random_state=42).tolist()
    print(f"\n--- Cluster {cluster_id} ---")
    for idea in sample:
        print(f"  • {idea}")


--- Cluster 0 ---
  • посмертное воздаяние и судьба души после смерти
  • Возможность искупления и посмертного покоя для грешной души
  • Посмертное воздаяние и судьба души после смерти
  • посмертное искупление и передача духовного наследия
  • загробное воздаяние как отражение земных деяний человека

--- Cluster 1 ---
  • Распад идентичности и границы между нормальностью и безумием
  • Зыбкость границы между реальностью и психозом
  • граница между нормальностью и безумием в экстремальных обстоятельствах
  • Зыбкость границы между нормальностью и безумием
  • Граница между героизмом и безумием

--- Cluster 2 ---
  • Ненависть как психологическая основа личности
  • ненависть как основа идентичности

--- Cluster 3 ---
  • нарциссизм и духовная незрелость как барьеры к подлинной близости
  • Неспособность к подлинной близости как следствие нарциссической самопоглощённости
  • Эгоизм и нарциссизм как препятствия к подлинной любви

--- Cluster 4 ---
  • Любовь как искусство, требующее м